# ❤️ Heart Disease Prediction - Detailed Project

This notebook demonstrates a complete machine learning pipeline for predicting **heart disease** using the classic UCI dataset. We go step by step:
- Exploratory Data Analysis (EDA)
- Data Preprocessing
- Logistic Regression
- Random Forest
- XGBoost
- Model Comparison
- Saving the Best Model

## 1. Import Libraries

Import all necessary libraries for data manipulation, visualization, machine learning models, and model persistence.

In [ ]:
# 1. Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset

Load the heart disease dataset (heart.csv) into a pandas DataFrame and display the first few rows.

In [ ]:
df = pd.read_csv("heart.csv")
df.head()

## 3. Exploratory Data Analysis (EDA)

Analyze the dataset structure, check for missing values, visualize the target distribution, and plot a correlation heatmap.

In [ ]:
print("Shape of dataset:", df.shape)
print("\nInfo:")
print(df.info())
print("\nSummary Stats:")
print(df.describe())
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Target distribution
sns.countplot(x='target', data=df, palette='Set2')
plt.title("Target Variable Distribution")
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 4. Split Features & Target

Separate the features and target variable, then split the data into training and testing sets.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 5. Feature Scaling

Apply standard scaling to the feature data using StandardScaler for improved model performance.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Logistic Regression

Train a logistic regression model, make predictions, and evaluate its performance with accuracy and classification report.

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

## 7. Random Forest

Train a random forest classifier, evaluate its performance, plot the confusion matrix, and visualize feature importances.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens")
plt.title("Confusion Matrix - Random Forest")
plt.show()

# Feature importance
feat_importances = pd.Series(rf.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh', color='orange')
plt.title("Random Forest Feature Importance")
plt.show()

## 8. XGBoost

Train an XGBoost classifier, evaluate its performance, plot the confusion matrix, and display feature importances.

In [ ]:
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

In [ ]:
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds")
plt.title("Confusion Matrix - XGBoost")
plt.show()

xgb.plot_importance(xgb_model)
plt.show()

## 9. Model Comparison

Compare the accuracy of all trained models and visualize the results using a bar plot.

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ]
})
print(results)

sns.barplot(x='Model', y='Accuracy', data=results, palette='viridis')
plt.title("Model Accuracy Comparison")
plt.show()

## 10. Save Best Model

Save the best performing model and the scaler to disk using pickle for future use.

In [ ]:
# Assume XGBoost is best
with open("best_model.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Best model and scaler saved successfully!")